# Championship Pipeline: LLM Classification Finetuning
**Model**: `microsoft/deberta-v3-base` (swap to `-large` for final sub)  
**Task**: 3-class LLM preference prediction (`winner_model_a` / `winner_model_b` / `winner_tie`)  
**Metric**: Log-loss - target `< 0.90` vs scratch-MLP baseline `~1.085`

---
| Stage | Description |
|-------|-------------|
| 1 | Environment & reproducibility |
| 2 | Data loading + schema validation |
| 3 | Stratified K-Fold assignment |
| 4 | Tokenization & Dataset |
| 5 | DeBERTa model definition |
| 6 | Cross-validation training loop |
| 7 | OOF evaluation |
| 8 | Test inference & ensemble averaging |
| 9 | Submission formatting + sanity checks |


In [ ]:
# Install / upgrade required packages (run once per Kaggle session)
import subprocess, sys
pkgs = [
    'transformers>=4.40.0',
    'sentencepiece',
    'protobuf',
]
for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('packages ready')


In [ ]:
"""
Championship Pipeline: LLM Classification Finetuning (Kaggle)
==============================================================
Model  : microsoft/deberta-v3-base  (swap to -large for final sub)
Task   : 3-class preference prediction (winner_model_a / _b / tie)
Metric : Log-loss (lower = better); target <0.90 vs baseline ~1.085
Pipeline stages
---------------
1. Environment & reproducibility
2. Data loading + schema validation
3. Stratified K-Fold assignment
4. Tokenization & Dataset
5. Model definition (DeBERTa + custom head)
6. Cross-validation training loop w/ checkpointing + metric logging
7. Out-of-fold (OOF) evaluation
8. Test inference & ensemble averaging
9. Submission formatting + sanity checks
"""
from __future__ import annotations


## 1. Environment & reproducibility


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1.  ENVIRONMENT & REPRODUCIBILITY


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
import gc
import math
import os
import random
import re
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModel,
    AutoTokenizer,
    get_cosine_schedule_with_warmup,
)
warnings.filterwarnings("ignore")
# ── Paths ──────────────────────────────────────────────────────────────────
_KAGGLE_WORKING = Path("/kaggle/working")
OUTPUT_DIR = _KAGGLE_WORKING if _KAGGLE_WORKING.is_dir() else Path.cwd()
CKPT_DIR = OUTPUT_DIR / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_SUBMISSION = str(OUTPUT_DIR / "submission.csv")
OUTPUT_OOF = str(OUTPUT_DIR / "oof_predictions.csv")
# ── Seed ───────────────────────────────────────────────────────────────────
SEED = 42
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False   # reproducibility > speed
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[env] device: {DEVICE}")
if torch.cuda.is_available():
    print(f"[env] gpu: {torch.cuda.get_device_name(0)}")
    print(f"[env] vram: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 2. Config


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.  CONFIG


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Swap MODEL_NAME to "microsoft/deberta-v3-large" for a ~0.02 ll improvement
# at the cost of ~2× VRAM and ~2× training time.
MODEL_NAME    = "microsoft/deberta-v3-base"
N_FOLDS       = 5
EPOCHS        = 3           # 3 epochs is usually sufficient for DeBERTa fine-tuning
BATCH_SIZE    = 8           # reduce to 4 for -large on 16GB VRAM
GRAD_ACCUM    = 4           # effective batch = BATCH_SIZE * GRAD_ACCUM = 32
MAX_LEN       = 512         # tokens per sample
LR            = 1e-5        # peak LR for AdamW
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.1         # fraction of total steps for LR warmup
LABEL_SMOOTH  = 0.05        # light label smoothing stabilises training
SWAP_AUGMENT  = True        # swap A<->B for 2× training data
FP16          = torch.cuda.is_available()   # mixed-precision
CLASSES       = ["winner_model_a", "winner_model_b", "winner_tie"]


## 3. Data loading & schema validation


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.  DATA LOADING & SCHEMA VALIDATION


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
def resolve_data_dir() -> str:
    need = {"train.csv", "test.csv", "sample_submission.csv"}
    candidates = [
        "/kaggle/input/competitions/llm-classification-finetuning",
        "/kaggle/input/llm-classification-finetuning",
    ]
    for d in candidates:
        if os.path.isdir(d) and need.issubset(set(os.listdir(d))):
            return d
    for dirpath, _, filenames in os.walk("/kaggle/input"):
        if need.issubset(set(filenames)):
            return dirpath
    raise FileNotFoundError(
        "Competition data not found. Attach 'llm-classification-finetuning' dataset."
    )
DATA_DIR = resolve_data_dir()
print(f"[log] DATA_DIR: {DATA_DIR}")
train_df   = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df    = pd.read_csv(os.path.join(DATA_DIR, "test.csv"),            dtype={"id": str})
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"), dtype={"id": str})
if "id" in train_df.columns:
    train_df["id"] = train_df["id"].astype(str)
# ── Schema checks ──────────────────────────────────────────────────────────
_required_train = {"prompt", "response_a", "response_b",
                   "winner_model_a", "winner_model_b", "winner_tie"}
_required_test  = {"id", "prompt", "response_a", "response_b"}
assert _required_train.issubset(train_df.columns), \
    f"Missing train columns: {_required_train - set(train_df.columns)}"
assert _required_test.issubset(test_df.columns), \
    f"Missing test columns: {_required_test - set(test_df.columns)}"
TEXT_COLS = ["prompt", "response_a", "response_b"]
for col in TEXT_COLS:
    train_df[col] = train_df[col].fillna("").astype(str)
    test_df[col]  = test_df[col].fillna("").astype(str)
# ── Canonical ID helper (handles int/float/str id variants) ───────────────
def canonical_id(series: pd.Series) -> pd.Series:
    def _one(v):
        if pd.isna(v):
            return "nan"
        try:
            fv = float(v)
            if np.isfinite(fv) and abs(fv - round(fv)) < 1e-9 * max(1.0, abs(fv)):
                return str(int(round(fv)))
        except (TypeError, ValueError):
            pass
        s = str(v).strip()
        if s.endswith(".0") and s[:-2].lstrip("-").isdigit():
            return s[:-2]
        return s
    return series.map(_one)
_k_test = set(canonical_id(test_df["id"]))
_k_sub  = set(canonical_id(sample_sub["id"]))
assert _k_test == _k_sub, (
    f"test vs sample_sub id mismatch - "
    f"only in test: {list(_k_test - _k_sub)[:5]}, "
    f"only in sub: {list(_k_sub - _k_test)[:5]}"
)
print(f"[log] train rows: {len(train_df):,}  |  test rows: {len(test_df):,}")
# ── Labels ─────────────────────────────────────────────────────────────────
def winner_cols_to_label(df: pd.DataFrame) -> np.ndarray:
    w = df[CLASSES].values.astype(np.float32)
    s = w.sum(axis=1)
    if not np.all(np.isclose(s, 1.0, atol=1e-3)):
        bad = np.where(~np.isclose(s, 1.0, atol=1e-3))[0]
        raise ValueError(f"winner columns must sum to 1 - bad rows: {bad[:10]}")
    return w.argmax(axis=1)
train_df["label"] = winner_cols_to_label(train_df)
print(f"[log] label distribution:\n{train_df['label'].value_counts().sort_index().to_string()}")


## 4. Fold assignment


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.  FOLD ASSIGNMENT  (deterministic, stratified)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
train_df["fold"] = -1
for fold, (_, val_idx) in enumerate(skf.split(train_df, train_df["label"])):
    train_df.loc[val_idx, "fold"] = fold
assert (train_df["fold"] == -1).sum() == 0, "Some rows not assigned a fold"
print(f"[log] {N_FOLDS}-fold split assigned (stratified by label)")


## 5. Tokenizer & Dataset


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5.  TOKENIZER & DATASET


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
print(f"[log] loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# Prompt template - keeps total tokens within MAX_LEN via truncation
# We give the full prompt and truncate the (longer) responses symmetrically.
def build_text(prompt: str, response_a: str, response_b: str) -> str:
    """
    Single-sequence format:
      [CLS] <prompt> [SEP] Response A: <response_a> [SEP] Response B: <response_b> [SEP]
    DeBERTa handles up to 512 tokens; we let the tokenizer truncate.
    """
    # Hard-cap raw text lengths before tokenization to stay under 512 tokens.
    # DeBERTa-v3 byte-pair encodes roughly 4 chars/token; 800 chars ~ 200 tokens.
    p = prompt[:600]
    ra = response_a[:900]
    rb = response_b[:900]
    return (
        f"{p}\n\n"
        f"### Response A\n{ra}\n\n"
        f"### Response B\n{rb}"
    )
class PreferenceDataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
        with_labels: bool,
        augment_swap: bool = False,
    ) -> None:
        rows = []
        for _, row in frame.iterrows():
            rows.append((
                build_text(row["prompt"], row["response_a"], row["response_b"]),
                int(row["label"]) if with_labels else -1,
            ))
            if augment_swap and with_labels:
                # Swapped: A<->B, labels 0<->1; tie stays 2
                swapped_label = {0: 1, 1: 0, 2: 2}[int(row["label"])]
                rows.append((
                    build_text(row["prompt"], row["response_b"], row["response_a"]),
                    swapped_label,
                ))
        self.texts  = [r[0] for r in rows]
        self.labels = [r[1] for r in rows]
        self.with_labels = with_labels
    def __len__(self) -> int:
        return len(self.texts)
    def __getitem__(self, idx: int):
        enc = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.with_labels:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


## 6. Model definition


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.  MODEL


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
class DeBERTaClassifier(nn.Module):
    """
    DeBERTa backbone + mean-pool of last hidden state + dropout + linear head.
    Mean-pool is more stable than [CLS] alone for long sequences.
    """
    def __init__(self, model_name: str, num_classes: int = 3, dropout: float = 0.1) -> None:
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size
        self.drop = nn.Dropout(dropout)
        self.head  = nn.Linear(hidden, num_classes)
        nn.init.normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)
    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor | None = None,
    ) -> torch.Tensor:
        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        out = self.backbone(**kwargs)
        # Mean-pool over non-padding positions
        hidden = out.last_hidden_state                          # B, L, H
        mask   = attention_mask.unsqueeze(-1).float()          # B, L, 1
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)  # B, H
        pooled = self.drop(pooled)
        return self.head(pooled)                               # B, 3


## 7. Loss function


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 7.  LOSS  (cross-entropy + label smoothing)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
class SmoothCrossEntropy(nn.Module):
    def __init__(self, smoothing: float = 0.05, num_classes: int = 3) -> None:
        super().__init__()
        self.smoothing   = smoothing
        self.num_classes = num_classes
    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        log_probs = F.log_softmax(logits, dim=-1)               # B, C
        with torch.no_grad():
            soft = torch.full_like(log_probs, self.smoothing / (self.num_classes - 1))
            soft.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        return -(soft * log_probs).sum(dim=-1).mean()
loss_fn = SmoothCrossEntropy(smoothing=LABEL_SMOOTH, num_classes=3)


## 8. Inference helper


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 8.  HELPERS


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def predict_loader(model: nn.Module, loader: DataLoader) -> np.ndarray:
    """Run inference; return softmax probabilities (N, 3)."""
    model.eval()
    probs_list = []
    for batch in loader:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        ttid = batch.get("token_type_ids")
        if ttid is not None:
            ttid = ttid.to(DEVICE)
        with autocast(enabled=FP16):
            logits = model(ids, mask, ttid)
        p = torch.softmax(logits.float(), dim=-1).cpu().numpy()
        probs_list.append(p)
    probs = np.vstack(probs_list)
    return probs / probs.sum(axis=1, keepdims=True)


## 9. Cross-validation training loop


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 9.  CROSS-VALIDATION TRAINING LOOP


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
oof_probs  = np.zeros((len(train_df), 3), dtype=np.float32)  # out-of-fold predictions
test_probs = np.zeros((len(test_df),  3), dtype=np.float32)  # accumulated test preds
fold_scores: list[float] = []
print(f"\n{'='*60}")
print(f"  Training {N_FOLDS}-fold CV  |  model: {MODEL_NAME}")
print(f"  epochs={EPOCHS}  batch={BATCH_SIZE}  accum={GRAD_ACCUM}  lr={LR}  fp16={FP16}")
print(f"{'='*60}\n")
for fold in range(N_FOLDS):
    print(f"\n── Fold {fold+1}/{N_FOLDS} ──────────────────────────────────────")
    set_seed(SEED + fold)  # per-fold seed for reproducibility
    tr_df  = train_df[train_df["fold"] != fold].reset_index(drop=True)
    val_df = train_df[train_df["fold"] == fold].reset_index(drop=True)
    val_orig_idx = train_df[train_df["fold"] == fold].index.tolist()
    # Datasets
    tr_ds  = PreferenceDataset(tr_df,  with_labels=True,  augment_swap=SWAP_AUGMENT)
    val_ds = PreferenceDataset(val_df, with_labels=True,  augment_swap=False)
    te_ds  = PreferenceDataset(test_df, with_labels=False, augment_swap=False)
    tr_loader  = DataLoader(tr_ds,  batch_size=BATCH_SIZE, shuffle=True,
                             num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                             num_workers=2, pin_memory=True)
    te_loader  = DataLoader(te_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                             num_workers=2, pin_memory=True)
    print(f"  train samples: {len(tr_ds):,}  |  val samples: {len(val_ds):,}")
    # Model, optimiser, scheduler
    model = DeBERTaClassifier(MODEL_NAME, num_classes=3, dropout=0.1).to(DEVICE)
    opt   = torch.optim.AdamW(
        [
            {"params": model.backbone.parameters(), "lr": LR},
            {"params": model.head.parameters(),     "lr": LR * 10},
        ],
        weight_decay=WEIGHT_DECAY,
    )
    total_steps   = math.ceil(len(tr_loader) / GRAD_ACCUM) * EPOCHS
    warmup_steps  = int(total_steps * WARMUP_RATIO)
    sched = get_cosine_schedule_with_warmup(opt, warmup_steps, total_steps)
    scaler = GradScaler(enabled=FP16)
    best_ll   = math.inf
    best_path = CKPT_DIR / f"fold{fold}_best.pt"
    for epoch in range(EPOCHS):
        # ── Train ────────────────────────────────────────────────────────
        model.train()
        running_loss = 0.0
        opt.zero_grad(set_to_none=True)
        for step, batch in enumerate(tr_loader, 1):
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            ttid = batch.get("token_type_ids")
            if ttid is not None:
                ttid = ttid.to(DEVICE)
            labs = batch["labels"].to(DEVICE)
            with autocast(enabled=FP16):
                logits = model(ids, mask, ttid)
                loss   = loss_fn(logits, labs) / GRAD_ACCUM
            scaler.scale(loss).backward()
            running_loss += loss.item() * GRAD_ACCUM
            if step % GRAD_ACCUM == 0 or step == len(tr_loader):
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                sched.step()
                opt.zero_grad(set_to_none=True)
        avg_tr_loss = running_loss / len(tr_loader)
        # ── Validate ─────────────────────────────────────────────────────
        val_p = predict_loader(model, val_loader)
        val_ll = log_loss(val_df["label"].values, val_p, labels=[0, 1, 2])
        print(
            f"  [epoch {epoch+1}/{EPOCHS}] "
            f"train_loss={avg_tr_loss:.4f}  val_log_loss={val_ll:.5f}",
            end="",
        )
        if val_ll < best_ll:
            best_ll = val_ll
            torch.save(model.state_dict(), best_path)
            print("  <- best", end="")
        print()
    # ── Load best checkpoint ─────────────────────────────────────────────
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    print(f"  [fold {fold+1}] best val_log_loss = {best_ll:.5f}")
    fold_scores.append(best_ll)
    # ── OOF predictions (no augment -> original val rows only) ────────────
    val_p = predict_loader(model, val_loader)
    oof_probs[val_orig_idx] = val_p
    # ── Test predictions (accumulated, averaged later) ───────────────────
    te_p = predict_loader(model, te_loader)
    test_probs += te_p / N_FOLDS
    # Free VRAM between folds
    del model, opt, sched, scaler, tr_ds, val_ds, tr_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()


## 10. OOF evaluation


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 10.  OOF EVALUATION


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
oof_probs_norm = oof_probs / oof_probs.sum(axis=1, keepdims=True)
overall_oof_ll = log_loss(train_df["label"].values, oof_probs_norm, labels=[0, 1, 2])
print(f"\n{'='*60}")
print(f"  CV Results")
print(f"{'='*60}")
for i, s in enumerate(fold_scores):
    print(f"  fold {i+1}: {s:.5f}")
print(f"  mean  : {np.mean(fold_scores):.5f}")
print(f"  std   : {np.std(fold_scores):.5f}")
print(f"  OOF   : {overall_oof_ll:.5f}   <- headline metric")
print(f"{'='*60}\n")
# Save OOF for stacking / meta-learning
oof_df = train_df[["id", "label", "fold"]].copy()
oof_df[CLASSES] = oof_probs_norm
oof_df.to_csv(OUTPUT_OOF, index=False)
print(f"[log] OOF predictions saved -> {OUTPUT_OOF}")


## 11. Submission formatting & sanity checks


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 11.  SUBMISSION FORMATTING & SANITY CHECKS


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
test_probs_norm = test_probs / test_probs.sum(axis=1, keepdims=True)
pred_tbl = pd.DataFrame({
    "_kid":          canonical_id(test_df["id"]).astype(str).values,
    CLASSES[0]:      test_probs_norm[:, 0],
    CLASSES[1]:      test_probs_norm[:, 1],
    CLASSES[2]:      test_probs_norm[:, 2],
})
pred_tbl = pred_tbl.groupby("_kid", as_index=False).first()
sub = sample_sub[["id"]].copy()
sub["_kid"] = canonical_id(sub["id"]).astype(str)
sub = sub.merge(pred_tbl, on="_kid", how="left").drop(columns=["_kid"])
# Check no missing cells
missing = sub[CLASSES].isna().any(axis=1).sum()
if missing:
    raise RuntimeError(f"submission has {missing} missing rows after merge")
# Normalise rows (float precision guard)
row_sums = sub[CLASSES].sum(axis=1).values
if not np.all(np.isclose(row_sums, 1.0, atol=1e-4)):
    p = sub[CLASSES].values.astype(np.float64)
    p = p / p.sum(axis=1, keepdims=True)
    sub[CLASSES] = p
sub.to_csv(OUTPUT_SUBMISSION, index=False)
# Final checks
_out = Path(OUTPUT_SUBMISSION)
assert _out.is_file(), f"Expected output not found: {OUTPUT_SUBMISSION}"
row_sums_final = sub[CLASSES].sum(axis=1).values
print("[check] missing_cells  :", 0)
print("[check] row_sum_ok_ratio:", float(np.mean(np.isclose(row_sums_final, 1.0, atol=1e-4))))
print(f"[check] submission rows : {len(sub):,}")
print(f"[check] file bytes      : {_out.stat().st_size:,}")
print(f"[log]   submission saved -> {OUTPUT_SUBMISSION}")
print(f"\n[FINAL] OOF log-loss = {overall_oof_ll:.5f}  (baseline was ~1.085)")
